In [5]:
from astropy.io import fits
import numpy as np
import pandas as pd

# File paths
filepath = r"C:\Users\prish\Research\SpenderQ\tiny_dr1\survey\catalogs\dr1\QSO\iron\QSO_cat_iron_cumulative_v0.fits"
outpath = r"C:\Users\prish\Research\Metal Absorbers/qso_candidates.csv"

# 1. Load FITS table
with fits.open(filepath) as hdul:
    data = hdul[1].data  # extension with TFIELDS = 68

# Convert to pandas DataFrame
df = pd.DataFrame(data)

# Keep only TARGETIDs that have at least one observation with 2.1 <= redshift <= 3.5
mask = (df['Z'] >= 2.1) & (df['Z'] <= 3.5)

mid_z_ids = df.loc[mask, 'TARGETID'].unique()

# Filter the DataFrame to only include those TARGETIDs
df = df[df['TARGETID'].isin(mid_z_ids)]

# 3. Filter 2: Remove ALL observations of a target if even one epoch has COADD_FIBERSTATUS != 0
bad_fiber_ids = df.loc[df['COADD_FIBERSTATUS'] != 0, 'TARGETID'].unique()
df = df[~df['TARGETID'].isin(bad_fiber_ids)]

# 1. Restrict to Stripe 82 Declination limits
df = df[(df['TARGET_DEC'] >= -1.25) & (df['TARGET_DEC'] <= 1.25)]

# 2. Slice RA 
# Start at RA = 0 and expand the RA limit
target_rows = 20000
ra_start = 0.0
ra_end = 0.5 

while True:
    patch_df = df[(df['TARGET_RA'] >= ra_start) & (df['TARGET_RA'] <= ra_end)]
    
    if len(patch_df) >= target_rows:
        break
    
    ra_end += 0.5  # Grow the RA box by 0.5 degrees and check again

df = patch_df

df = df.sort_values('TARGETID')

# 5. Save the final processed DataFrame to CSV
df.to_csv(outpath, index=False)

# 6. Display final summary metrics
print("Saved metal absorber candidates to:", outpath)
print("Total rows written:", len(df))

num_unique_targets = df['TARGETID'].nunique()
print("--------------------------------------------------")
print("Number of unique TARGETIDs remaining:", num_unique_targets)
print("--------------------------------------------------")

Saved metal absorber candidates to: C:\Users\prish\Research\Metal Absorbers/qso_candidates.csv
Total rows written: 20040
--------------------------------------------------
Number of unique TARGETIDs remaining: 10228
--------------------------------------------------
